# 1. Base Setup

## 1.1 Install packages

In [13]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [14]:
!pip install kagglehub
!pip install statsmodels
!pip install xgboost

## 1.2 Load all needed imports

In [15]:
from pathlib import Path

import pandas as pd
import kagglehub
from kagglehub import KaggleDatasetAdapter
import numpy as np

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.metrics import log_loss, accuracy_score, f1_score

from sklearn.model_selection import PredefinedSplit

import sys, os
sys.path.append(os.path.abspath(".."))

# 2. Data Cleaning and preprocessing


In [16]:
import cf_copilot
print(cf_copilot.__file__)

/home/jeroenewalts/code/EwaltsJ/cf_copilot/cf_copilot/__init__.py


In [17]:
def load_cashflow_data(csv_name: str = "dataset.csv") -> pd.DataFrame:
    """Load invoice dataset from local raw_data folder, or download from Kaggle.

    Args:
        csv_name: filename of the CSV to load.

    Returns:
        A pandas DataFrame with the raw invoice data.
    """
    base_dir = Path.cwd().parent
    raw_data_dir = base_dir / "raw_data"
    raw_data_dir.mkdir(parents=True, exist_ok=True)
    local_path = raw_data_dir / csv_name

    if local_path.is_file():
        print(f"Loading local file: {local_path}")
        return pd.read_csv(local_path)

    print("Local file not found, downloading from Kaggle via kagglehub...")
    df = kagglehub.dataset_load(
        KaggleDatasetAdapter.PANDAS,
        "pradumn203/payment-date-prediction-for-invoices-dataset",
        "dataset.csv",
    )

    df.to_csv(local_path, index=False)
    print(f"Saved local copy to {local_path}")

    return df

In [18]:
from cf_copilot.ml_logic.data import data_cleaning, build_sliding_window_snapshots
from cf_copilot.ml_logic.encoders import preprocess, NUMERIC_FEATURES, CATEGORICAL_FEATURES

hist_df = load_cashflow_data()

df = data_cleaning(hist_df)
big_df = build_sliding_window_snapshots(df)

big_df = big_df.sort_values("invoice_sent").reset_index(drop=True)

# Train+valid for search, final holdout for test
valid_cutoff = big_df["reference_date"].quantile(0.6)
test_cutoff = big_df["reference_date"].quantile(0.8)

search_df = big_df[big_df["reference_date"] <= test_cutoff].copy()
final_test_df = big_df[big_df["reference_date"] > test_cutoff].copy()

# Explicit train / validation split inside search_df
train_df = search_df[search_df["reference_date"] <= valid_cutoff].copy()
valid_df = search_df[search_df["reference_date"] > valid_cutoff].copy()

X_train_xgb, y_train_xgb = preprocess(train_df)
X_valid_xgb, y_valid_xgb = preprocess(valid_df)
X_final_test_xgb, y_final_test_xgb = preprocess(final_test_df)

# XGBoost needs zero-based labels
y_train_xgb = y_train_xgb - 1
y_valid_xgb = y_valid_xgb - 1
y_final_test_xgb = y_final_test_xgb - 1

print("Train:", X_train_xgb.shape, y_train_xgb.shape)
print("Valid:", X_valid_xgb.shape, y_valid_xgb.shape)
print("Test :", X_final_test_xgb.shape, y_final_test_xgb.shape)

Loading local file: /home/jeroenewalts/code/EwaltsJ/cf_copilot/raw_data/dataset.csv
Original rows: 39152
Augmented rows: 98169
week_bucket
1    38825
2    33060
3    10401
4     4009
5     3172
6     2192
7     6510
Name: count, dtype: int64
Train: (60018, 18) (60018,)
Valid: (19707, 18) (19707,)
Test : (18444, 18) (18444,)


In [19]:
numeric_transformer = SimpleImputer(strategy="median")

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value=-1)),
    ("encoder", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)),
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, NUMERIC_FEATURES),
        ("cat", categorical_transformer, CATEGORICAL_FEATURES),
    ],
    remainder="drop",
)

preprocessor

ColumnTransformer(transformers=[('num', SimpleImputer(strategy='median'),
                                 ['business_year', 'invoice_age_days',
                                  'days_until_due', 'pay_terms_days',
                                  'invoice_month', 'due_month', 'days_past_due',
                                  'customer_avg_delay', 'late_payment_ratio',
                                  'prev_transaction_count',
                                  'days_since_last_invoice',
                                  'customer_risk_score', 'invoice_amount',
                                  'invoice_amount_log']),
                                ('cat',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(fill_value=-1,
                                                                strategy='constant')),
                                                 ('encoder',
                                                  OrdinalEncoder(handle_unknown='use_encoded_value',
                                                                 unknown_value=-1))]),
                                 ['invoice_currency', 'document_type',
                                  'invoice_size_cat', 'invoice_size_cat_q'])])

In [20]:
from xgboost import XGBClassifier
import datetime

xgb_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", XGBClassifier(
        objective="multi:softprob",
        num_class=7,
        eval_metric="mlogloss",
        random_state=42,
        n_jobs=4,
        tree_method="hist",
        verbosity=0,
        colsample_bytree=0.7265477506155757,
        gamma=0.058794858725743554,
        learning_rate=0.024522728891053808,
        max_depth=7,
        min_child_weight=2,
        n_estimators=352,
        reg_alpha=0.5867511656638482,
        reg_lambda=1.930510614528276,
        subsample=0.8821102743060054,
    ))
])

print("START RAW FIT:", datetime.datetime.now())
best_model = xgb_pipeline.fit(X_train_xgb, y_train_xgb)
print("END RAW FIT:", datetime.datetime.now())

START RAW FIT: 2026-03-25 13:15:28.234736
END RAW FIT: 2026-03-25 13:15:32.075741


In [21]:
from sklearn.calibration import CalibratedClassifierCV
from sklearn.frozen import FrozenEstimator

print("START CALIBRATION:", datetime.datetime.now())

calibrated_model = CalibratedClassifierCV(
    estimator=FrozenEstimator(best_model),
    method="sigmoid"
)

calibrated_model.fit(X_valid_xgb, y_valid_xgb)

print("END CALIBRATION:", datetime.datetime.now())

START CALIBRATION: 2026-03-25 13:15:32.095254
END CALIBRATION: 2026-03-25 13:15:32.425740


In [22]:
y_proba_raw = best_model.predict_proba(X_final_test_xgb)
y_pred_raw = best_model.predict(X_final_test_xgb)

clf_classes_raw = np.array(best_model.named_steps["classifier"].classes_)

raw_log_loss = log_loss(y_final_test_xgb, y_proba_raw, labels=clf_classes_raw)
raw_accuracy = accuracy_score(y_final_test_xgb, y_pred_raw)
raw_f1_macro = f1_score(y_final_test_xgb, y_pred_raw, average="macro")

print("\nRaw XGBoost holdout performance")
print(f"log_loss  : {raw_log_loss:.6f}")
print(f"accuracy  : {raw_accuracy:.6f}")
print(f"f1_macro  : {raw_f1_macro:.6f}")


Raw XGBoost holdout performance
log_loss  : 0.718217
accuracy  : 0.751572
f1_macro  : 0.564193


In [23]:
y_proba_cal = calibrated_model.predict_proba(X_final_test_xgb)
y_pred_cal = calibrated_model.predict(X_final_test_xgb)

clf_classes_cal = np.array(calibrated_model.classes_)

cal_log_loss = log_loss(y_final_test_xgb, y_proba_cal, labels=clf_classes_cal)
cal_accuracy = accuracy_score(y_final_test_xgb, y_pred_cal)
cal_f1_macro = f1_score(y_final_test_xgb, y_pred_cal, average="macro")

print("\nCalibrated XGBoost holdout performance")
print(f"log_loss  : {cal_log_loss:.6f}")
print(f"accuracy  : {cal_accuracy:.6f}")
print(f"f1_macro  : {cal_f1_macro:.6f}")


Calibrated XGBoost holdout performance
log_loss  : 0.774510
accuracy  : 0.750596
f1_macro  : 0.563692


In [24]:
from cf_copilot.cashflow_prediction.evaluation import build_actual_weekly_cf, compare_forecast_vs_actual, compute_forecast_metrics
from cf_copilot.cashflow_prediction.registry import WEEK_CLASSES
from colorama import Fore, Style
from cf_copilot.ml_logic.encoders import preprocess


def evaluate_forecast_holdout(
    model: Pipeline,
    test_df: pd.DataFrame,
    verbose: bool = True
) -> tuple[dict, dict]:
    """
    Evaluate forecast quality on the holdout set by reference_date.

    Args:
        model: fitted sklearn Pipeline
        test_df: holdout snapshot dataframe
        verbose: whether to print evaluation output

    Returns:
        forecast_metrics: scalar metrics to merge into MLflow metrics
        forecast_summary: JSON-serializable artifact with per-reference-date detail
    """
    if verbose:
        print(Fore.BLUE + f"\nEvaluating forecast on {len(test_df)} rows..." + Style.RESET_ALL)

    if model is None:
        if verbose:
            print("❌ No model to evaluate forecast")
        return {}, {}

    if len(test_df) == 0:
        if verbose:
            print("❌ No holdout rows available for forecast evaluation")
        return {}, {}

    per_reference_results = []
    if hasattr(model, "named_steps") and "classifier" in model.named_steps:
        model_classes = np.array(model.named_steps["classifier"].classes_)
    else:
        model_classes = np.array(model.classes_)

    # Support both label schemes:
    # standard models: 1..7
    # XGBoost: 0..6
    if np.array_equal(np.sort(model_classes), np.array([0, 1, 2, 3, 4, 5, 6])):
        class_to_index = {int(c) + 1: i for i, c in enumerate(model_classes)}
    else:
        class_to_index = {int(c): i for i, c in enumerate(model_classes)}

    for reference_date, snapshot_df in test_df.groupby("reference_date"):
        snapshot_df = snapshot_df.copy()
        if len(snapshot_df) == 0:
            continue

        X_snapshot, _ = preprocess(snapshot_df)
        probas = model.predict_proba(X_snapshot)

        pred_cash_df = snapshot_df.copy()

        for w in WEEK_CLASSES:
            pred_cash_df[f"p_{w}"] = (
                probas[:, class_to_index[int(w)]]
                if int(w) in class_to_index
                else 0.0
            )

        weekly_forecast_df = pd.DataFrame([
            {
                "week_bucket": int(w),
                "forecast_cash": round(
                    float((pred_cash_df["total_open_amount"] * pred_cash_df[f"p_{w}"]).sum()),
                    2,
                ),
            }
            for w in WEEK_CLASSES
        ])

        total_invoice_amount = round(float(pred_cash_df["total_open_amount"].sum()), 2)
        total_expected_cash = round(float(weekly_forecast_df["forecast_cash"].sum()), 2)

        if total_expected_cash - total_invoice_amount > 0:
            print("Expected cash exceeds total invoice amount. This should not happen.")

        actual_weekly_df = build_actual_weekly_cf(
            invoices_df=snapshot_df,
            reference_date=pd.Timestamp(reference_date),
        )

        comparison_df = compare_forecast_vs_actual(
            weekly_forecast_df=weekly_forecast_df,
            actual_weekly_df=actual_weekly_df,
        )

        snapshot_metrics = compute_forecast_metrics(comparison_df)

        per_reference_results.append({
            "reference_date": str(pd.Timestamp(reference_date).date()),
            "forecast_check": {
                "total_invoice_amount": total_invoice_amount,
                "total_expected_cash": total_expected_cash,
            },
            "forecast_metrics": {
                "forecast_mae_weekly": float(snapshot_metrics["MAE (weekly)"]),
                "forecast_mape_weekly": (
                    float(snapshot_metrics["MAPE (weekly)"])
                    if pd.notna(snapshot_metrics["MAPE (weekly)"])
                    else None
                ),
                "total_actual_cf": float(snapshot_metrics["Total actual cf"]),
                "total_forecast_cf": float(snapshot_metrics["Total forecast cf"]),
                "total_cf_difference": float(snapshot_metrics["Total cf difference"]),
            },
        })

    if not per_reference_results:
        if verbose:
            print("❌ No per-reference-date forecast results available")
        return {}, {}

    mae_values = [r["forecast_metrics"]["forecast_mae_weekly"] for r in per_reference_results]
    mape_values = [
        r["forecast_metrics"]["forecast_mape_weekly"]
        for r in per_reference_results
        if r["forecast_metrics"]["forecast_mape_weekly"] is not None
    ]
    total_actual_values = [r["forecast_metrics"]["total_actual_cf"] for r in per_reference_results]
    total_forecast_values = [r["forecast_metrics"]["total_forecast_cf"] for r in per_reference_results]
    total_diff_values = [r["forecast_metrics"]["total_cf_difference"] for r in per_reference_results]

    forecast_metrics = {
        "forecast_mae_weekly": float(np.mean(mae_values)),
        "forecast_mape_weekly": float(np.mean(mape_values)) if mape_values else np.nan,
        "forecast_total_actual_cf": float(np.mean(total_actual_values)),
        "forecast_total_forecast_cf": float(np.mean(total_forecast_values)),
        "forecast_total_cf_difference": float(np.mean(total_diff_values)),
    }

    forecast_summary = {
        "per_reference_date": per_reference_results,
        "aggregate": {
            "forecast_mae_weekly": forecast_metrics["forecast_mae_weekly"],
            "forecast_mape_weekly": (
                forecast_metrics["forecast_mape_weekly"]
                if pd.notna(forecast_metrics["forecast_mape_weekly"])
                else None
            ),
            "forecast_total_actual_cf": forecast_metrics["forecast_total_actual_cf"],
            "forecast_total_forecast_cf": forecast_metrics["forecast_total_forecast_cf"],
            "forecast_total_cf_difference": forecast_metrics["forecast_total_cf_difference"]
        },
    }

    if verbose:
        print(f"✅ Forecast MAE weekly: {forecast_metrics['forecast_mae_weekly']:.2f}")

        if pd.notna(forecast_metrics["forecast_mape_weekly"]):
            print(f"✅ Forecast MAPE weekly: {forecast_metrics['forecast_mape_weekly']:.4f}")
        else:
            print("✅ Forecast MAPE weekly: None")

        print(f"✅ Forecast total actual cf: {forecast_metrics['forecast_total_actual_cf']:.2f}")
        print(f"✅ Forecast total forecast cf: {forecast_metrics['forecast_total_forecast_cf']:.2f}")
        print(f"✅ Forecast total cf difference: {forecast_metrics['forecast_total_cf_difference']:.2f}")

    return forecast_metrics, forecast_summary

In [25]:
forecast_metrics_raw, forecast_summary_raw = evaluate_forecast_holdout(
    model=best_model,
    test_df=final_test_df,
    verbose=True,
)

print("\nRaw XGBoost forecast metrics")
for k, v in forecast_metrics_raw.items():
    print(f"{k}: {v}")

pd.DataFrame([forecast_metrics_raw])


Evaluating forecast on 18444 rows...
✅ Forecast MAE weekly: 512759.37
✅ Forecast MAPE weekly: 0.2910
✅ Forecast total actual cf: 27906118.13
✅ Forecast total forecast cf: 27717441.78
✅ Forecast total cf difference: -188676.35

Raw XGBoost forecast metrics
forecast_mae_weekly: 512759.36950000003
forecast_mape_weekly: 0.29100000000000004
forecast_total_actual_cf: 27906118.129499994
forecast_total_forecast_cf: 27717441.776500005
forecast_total_cf_difference: -188676.353


,forecast_mae_weekly,forecast_mape_weekly,forecast_total_actual_cf,forecast_total_forecast_cf,forecast_total_cf_difference
0,512759.3695,0.291,2.790612e+07,2.771744e+07,-188676.353


In [26]:
forecast_metrics_cal, forecast_summary_cal = evaluate_forecast_holdout(
    model=calibrated_model,
    test_df=final_test_df,
    verbose=True,
)

print("\nCalibrated XGBoost forecast metrics")
for k, v in forecast_metrics_cal.items():
    print(f"{k}: {v}")

pd.DataFrame([forecast_metrics_cal])


Evaluating forecast on 18444 rows...
✅ Forecast MAE weekly: 704618.74
✅ Forecast MAPE weekly: 0.4885
✅ Forecast total actual cf: 27906118.13
✅ Forecast total forecast cf: 27875179.98
✅ Forecast total cf difference: -30938.15

Calibrated XGBoost forecast metrics
forecast_mae_weekly: 704618.7380000001
forecast_mape_weekly: 0.4885
forecast_total_actual_cf: 27906118.129499994
forecast_total_forecast_cf: 27875179.983499993
forecast_total_cf_difference: -30938.146


,forecast_mae_weekly,forecast_mape_weekly,forecast_total_actual_cf,forecast_total_forecast_cf,forecast_total_cf_difference
0,704618.738,0.4885,2.790612e+07,2.787518e+07,-30938.146


In [ ]:
comparison_df = pd.DataFrame([
    {
        "model": "XGBoost_raw",
        "log_loss": raw_log_loss,
        "accuracy": raw_accuracy,
        "f1_macro": raw_f1_macro,
        "forecast_mae_weekly": forecast_metrics_raw["forecast_mae_weekly"],
        "forecast_mape_weekly": forecast_metrics_raw["forecast_mape_weekly"],
        "forecast_total_cf_difference": forecast_metrics_raw["forecast_total_cf_difference"],
    },
    {
        "model": "XGBoost_calibrated_sigmoid",
        "log_loss": cal_log_loss,
        "accuracy": cal_accuracy,
        "f1_macro": cal_f1_macro,
        "forecast_mae_weekly": forecast_metrics_cal["forecast_mae_weekly"],
        "forecast_mape_weekly": forecast_metrics_cal["forecast_mape_weekly"],
        "forecast_total_cf_difference": forecast_metrics_cal["forecast_total_cf_difference"],
    },
])

comparison_df

,model,log_loss,accuracy,f1_macro,forecast_mae_weekly,forecast_mape_weekly,forecast_total_cf_difference
0,XGBoost_raw,0.718217,0.751572,0.564193,512759.3695,0.2910,-188676.353
1,XGBoost_calibrated_sigmoid,0.774510,0.750596,0.563692,704618.7380,0.4885,-30938.146
